# Module 2: LoRA Fine-Tuning with Vertex AI

In this notebook, we will demonstrate how to perform Parameter-Efficient Fine-Tuning (PEFT) using **Low-Rank Adaptation (LoRA)** on an open model (e.g., Llama 3.1 8B or Gemma 3) using Vertex AI.

We will also showcase two key Vertex AI differentiators:
1. **Vertex AI Experiments**: To track and manage our tuning jobs, hyperparameters, and metadata.
2. **Managed TensorBoard**: To visualize the training metrics (loss, learning rate) in real-time.

In [ ]:
.pip install google-cloud-aiplatform python-dotenv -q

## 1. Setup and Authentication

In [ ]:
import os
from google.cloud import aiplatform
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

PROJECT_ID = os.getenv("PROJECT_ID", "YOUR_PROJECT_ID")
LOCATION = os.getenv("LOCATION", "us-central1") # Tuning is broadly supported in us-central1 and europe-west4
STAGING_BUCKET = os.getenv("STAGING_BUCKET", "YOUR_GCS_BUCKET_NAME")

# Ensure bucket has gs:// prefix for Vertex SDK
if not STAGING_BUCKET.startswith("gs://"):
    STAGING_BUCKET = f"gs://{STAGING_BUCKET}"

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)

print(f"Project ID: {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Staging Bucket: {STAGING_BUCKET}")

## 2. Initialize Vertex Experiment & TensorBoard

First, we create a managed TensorBoard instance if we don't have one. Then we initialize a Vertex Experiment to capture the context of our tuning runs.

In [ ]:
EXPERIMENT_NAME = os.getenv("EXPERIMENT_NAME", "cyber-tuning-demo")
TENSORBOARD_NAME = "cyber-tuning-tb"

# 1. Create or get existing TensorBoard instance
tb_instances = aiplatform.Tensorboard.list(filter=f'display_name="{TENSORBOARD_NAME}"')
if tb_instances:
    tb_instance = tb_instances[0]
    print(f"Found existing TensorBoard: {tb_instance.resource_name}")
else:
    print(f"Creating new TensorBoard instance: {TENSORBOARD_NAME} ..")
    tb_instance = aiplatform.Tensorboard.create(display_name=TENSORBOARD_NAME)
    print(f"Created TensorBoard: {tb_instance.resource_name}")

# 2. Initialize the Experiment
aiplatform.init(experiment=EXPERIMENT_NAME, experiment_tensorboard=tb_instance)
print(f"Initialized Experiment: {EXPERIMENT_NAME}")

## 3. Configure and Launch LoRA Tuning Job

We use the `aiplatform.PipelineJob` (via the Vertex AI SDK's tuning capabilities or the Generative AI tuning wrapper) to launch a supervised fine tuning job.

*Note: Update the `TRAIN_DATA_URI` with the output from Module 1.*

In [ ]:
from vertexai.tuning import sft

# ==========================================================================
# PRE-REQUISITE: Paste the URI from demo_01 output below
TRAIN_DATA_URI = "gs://YOUR_BUCKET/tuning_data/lora_train.jsonl"
# ==========================================================================

# Pick a supported model from Vertex AI model garden, e.g., Llama 3.1 8B
BASE_MODEL = "google/gemma-3-1b-it"

# Define LoRA specific hyperparameters
# LoRA rank controls the dimensionality of the update matrices. Lower = smaller/faster, Higher = more capacity.
lora_hyperparams = {
    "lora_rank": 16, 
    "lora_alpha": 32,
    "learning_rate": 2e-4,
    "epochs": 1,
}

print("Starting LoRA Tuning Job..")

# Start a new run context in the experiment
aiplatform.start_run("run-lora-1")
aiplatform.log_params(lora_hyperparams)
aiplatform.log_params({"tuning_mode": "LoRA", "base_model": BASE_MODEL})

try:
    # Note: Vertex AI Tuning SDK simplifies this heavily
    sft_tuning_job = sft.train(
        source_model=BASE_MODEL,
        train_dataset=TRAIN_DATA_URI,
        # The adapter size and parameters dictate it is a LoRA/PEFT job vs Full.
        # Under the hood for supported open models, specifying adapter size triggers PEFT.
        adapter_size="16", 
        learning_rate_multiplier=1.0,
        epochs=lora_hyperparams["epochs"],
    )
    
    print("Tuning job submitted. You can view it in the Vertex AI Console under 'Pipelines' or 'Model Registry'.")
    print(f"Job Name: {sft_tuning_job.name}")
    
except Exception as e:
    print(f"Error submitting job: {e}")
    print("Please ensure the TRAIN_DATA_URI is correct and the necessary permissions are granted.")

finally:
    # End the experiment run
    aiplatform.end_run()
    print("Experiment run ended.")

## 4. Monitor via TensorBoard

Once the pipeline job begins training, you can click on the TensorBoard link to watch the loss curves drop in real-time.

In [ ]:
print("Your TensorBoard URL is:")
print(tb_instance.resource_name)
print("\nAlternatively, navigate to Vertex AI -> Experiments -> TensorBoard Instances in the Google Cloud Console.")